In [21]:
import pandas as pd
import openpyxl

In [22]:
df = pd.read_excel("~/sno_data/raw/xu.xlsx", header = 0, sheet_name = "Table S2")

# this study performed FDR filtering < 0.01 to peptides prior to reporting 
# repeated experiments 3 times too - think about whether to filter on this - i feel showing up once is enough
# just need to parse the site column
print(df.head())

     #            Sequence Protein accession  Gene  \
0  1.0      LSDGELLFECAGHK            C9JG97  AAMP   
1  2.0       AVYTQDCPLAAAK            P49588  AARS   
2  3.0               ELACK            P49588  AARS   
3  4.0  MFVEEVSTGQECGVVLDK            P49588  AARS   
4  NaN  MFVEEVSTGQECGVVLDK            P49588  AARS   

                                       Modifications Site assignment      #1  \
0  N-Term(iTRAQ8plex); C10(iTRAQ-8plex); K14(iTRA...       C10 (SNO)  1.0070   
1  N-Term(iTRAQ8plex); C7(iTRAQ-8plex); K13(iTRAQ...        C7 (SNO)  0.7810   
2  N-Term(iTRAQ8plex); C4(iTRAQ-8plex); K5(iTRAQ8...        C4 (SNO)  1.2180   
3  N-Term(iTRAQ8plex); C12(iTRAQ-8plex); K18(iTRA...       C12 (SNO)  0.9375   
4  N-Term(iTRAQ8plex); M1(Oxidation); C12(iTRAQ-8...       C12 (SNO)  0.9860   

         #2        #3  Charge  ...  Unnamed: 17  Unnamed: 18  Unnamed: 19  \
0       NaN  0.990000       4  ...          NaN          NaN          NaN   
1  0.962443  0.926667       4  ...      

In [ ]:
import re
import os 


def read_fasta(path):
    seqs, acc = {}, None
    with open(os.path.expanduser(path)) as f:     # CHANGED: open() doesn't expand "~"
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                acc = line.split("|")[1]          # >sp|P12345|NAME_HUMAN ... -> P12345
                seqs[acc] = []
            elif acc:
                seqs[acc].append(line)
    return {k: "".join(v) for k, v in seqs.items()}

proteome = read_fasta("~/human_proteome/proteome/human_proteome.fasta")



rows = []
missing, no_match, multi_match, bad_residue = [], [], [], []
sno_dict = {}
for id, row in df.iterrows():

    uniprot = row["Protein accession"]
    seq = row["Sequence"]
    
    parts = row["Site assignment"].split(";")
    parts = [part.strip() for part in parts]
    idx_list = []    
    for i in range(0, len(parts)):

        if "(SNO)" in parts[i]:
            to_check = parts[i]
            match = re.search(r'C(\d+)', to_check)
            if match:
                position = int(match.group(1))
                #print(position)
                idx = position - 1
                if idx >= len(seq) or seq[idx].upper() != "C":
                    bad_residue.append((uniprot, seq, part))
                    continue
                idx_list.append(idx)
                #print(len(idx_list))


    if not idx_list:
        continue

    # if len(idx_list) > 1:
    #     print(id)
    
    sno_dict[uniprot] = idx_list

    accessions = [a.strip() for a in str(uniprot).split(";") if a.strip()]
    candidates = [a for a in accessions if a in proteome]
    if not candidates:
        missing.append(uniprot)
        continue

    pattern = re.sub(r"[IL]", "[IL]", seq.upper())
    matched = False                                    # NEW

    for acc in candidates:                             # NEW: try every accession in the group
        prot_seq = proteome[acc]
        starts = [m.start() for m in re.finditer(f"(?={pattern})", prot_seq)]
        if len(starts) == 0:
            continue
        if len(starts) > 1:
            multi_match.append((acc, seq))
            matched = True
            continue
        matched = True
        for idx in idx_list:
            rows.append({"id": acc, "position": starts[0] + idx + 1})   # CHANGED: acc, not the group string

    if not matched:                                    # NEW
        no_match.append((uniprot, seq))
print(len(missing))
print(missing)







print(f"{len(out)} sites written")
print(f"missing from FASTA: {len(missing)}, no match: {len(no_match)}, "
      f"multi match: {len(multi_match)}, bad residue: {len(bad_residue)}")

# match = re.search(r'C(\d+)', text)

# if match:
#     number = match.group(1)
#     print(number)


262
['C9JG97', 'B6VEX4;A0A0A0MRT6', 'E7EQT4', 'E7EQT4', 'K7EM38', 'K7EM38', 'H7C2M6', 'H0YBE5', 'A0A0C4DG85', 'F8W1A4', 'H0Y804', 'H0Y804', 'R4GN94', 'A0A0A0MSK8', 'H0Y7C1', 'H0Y7C1', 'R4GN72', 'R4GN72', 'E7ES00', 'B3KSI3', 'A0A087WWW9', 'A0A0J9YWX1', 'C9JQV0', 'F8VPD4', 'F8VPD4', 'F8VPD4', 'F8VPD4', 'F8VPD4', 'D6R938;E9PBG7', 'B1AK87', 'G3V169', 'G3V169', 'A0A0C4DGB5;E7EVY3', 'A0A0C4DGB5;E7EVY3;H0YA91;F8W7E0', 'C9JQ41', 'H3BLV0', 'H3BLV0', 'H3BLV0', 'H3BLV0', 'H3BLV0', 'H3BLV0', 'E9PNW4', 'A6NNI4', 'A0A0A0MRJ0', 'Q5QPR3;Q5QPR4', 'A0A087WWV2', 'G3V1A4', 'G3V1A4', 'G3V1A4', 'G3V1A4', 'G3V1A4', 'F5GWX5', 'K7EMY3', 'J3KSJ7', 'H0YB20', 'R4GMX3', 'A0A087X0I4', 'E7EM64', 'H7C4E5', 'F2Z2V0', 'F2Z2V0', 'B4DJV2', 'O15320;G3V599', 'B4DGU4', 'B4DGU4', 'C9JZR2', 'C9JZR2', 'K7EK33;K7EQ02', 'E7EX90', 'E7EX90', 'H3BLZ8', 'H3BLZ8', 'I3L0H8', 'I3L0H8', 'G3V529', 'A0A0D9SFB3', 'A0A0D9SFB3', 'A0A0G2JH68;A0A140T8Z0', 'H0YNW5', 'H0YNW5', 'A0A0C4DGA2', 'A0A0C4DGA2', 'C9J4W5', 'C9J4W5', 'C9J4W5', 'C9J4W5', '

In [121]:
#give organism column
out = pd.DataFrame(rows).drop_duplicates()
out["organism"] = "Homo sapiens"
# write the CSV
out.to_csv("/home/callum/sno_data/processed/xu/xu_org.csv", index=False)

In [119]:
print(out[out["id"].duplicated(keep=False)])


print(df[df["Protein accession"] == "P49588"])

          id  position organism
0     P49588       671    human
1     P49588       115    human
2     P49588       525    human
4     P49588       533    human
5     P61221        38    human
...      ...       ...      ...
2134  P63104       189    human
2135  P63104        94    human
2136  P63104        25    human
2144  Q8N988       203    human
2145  Q8N988       206    human

[1248 rows x 3 columns]
     #             Sequence Protein accession  Gene  \
1  2.0        AVYTQDCPLAAAK            P49588  AARS   
2  3.0                ELACK            P49588  AARS   
3  4.0   MFVEEVSTGQECGVVLDK            P49588  AARS   
4  NaN   MFVEEVSTGQECGVVLDK            P49588  AARS   
5  5.0  TCFYAEQGGQIYDEGYLVK            P49588  AARS   

                                       Modifications Site assignment      #1  \
1  N-Term(iTRAQ8plex); C7(iTRAQ-8plex); K13(iTRAQ...        C7 (SNO)  0.7810   
2  N-Term(iTRAQ8plex); C4(iTRAQ-8plex); K5(iTRAQ8...        C4 (SNO)  1.2180   
3  N-Term(iTRAQ8plex

In [116]:
for i, k in sno_dict.items():
    if len(k) > 1:
        print(k)

[7, 10]
[7, 10, 11, 16]
[3, 8]
[12, 17]
[7, 10]
[11, 17]
[1, 9]
[6, 17]
[4, 6]
[9, 12]


In [112]:
idxs = [237, 342, 1313, 1544, 2174]

for i in idxs:
    print(df[["Sequence", "Site assignment"]].iloc[i])

for k, i in sno_dict.items():
    if k == "APGARPACYSCCPAHSCTASCAR":
        print(f"{k}, {i}")



Sequence              LTALAGGCVMCVLGRR
Site assignment    C8 (SNO); C11 (SNO)
Name: 237, dtype: str
Sequence                             APGARPACYSCCPAHSCTASCAR
Site assignment    C8 (SNO); C11 (SNO); C12 (SNO); C17 (SNO)
Name: 342, dtype: str
Sequence                  ITFGFISCPICK
Site assignment    C8 (SNO); C11 (SNO)
Name: 1313, dtype: str
Sequence                        GCITIIGGGDTATCCAK
Site assignment    C2 (SNO); C14 (SNO); C15 (SNO)
Name: 1544, dtype: str
Sequence           XGTLEECQHQTNWPNHMCEK
Site assignment     C7 (SNO); C18 (SNO)
Name: 2174, dtype: str


In [87]:
with open("/home/callum/human_proteome/proteome/human_proteome.fasta", "r") as f:
    for line in f:
        print(line)


>sp|A0A0A0MRZ8|KVD11_HUMAN Immunoglobulin kappa variable 3D-11 OS=Homo sapiens OX=9606 GN=IGKV3D-11 PE=3 SV=6

MEAPAQLLFLLLLWLPDTTGEIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKP

GQAPRLLIYDASNRATGIPARFSGSGPGTDFTLTISSLEPEDFAVYYCQQRSNWH

>sp|A8MUZ8|Z705G_HUMAN Putative zinc finger protein 705G OS=Homo sapiens OX=9606 GN=ZNF705G PE=2 SV=2

MHSLKKLTFEDVAIDFTQEEWAMMDTSKRKLYRDVMLENISHLVSLGYQISKSYIILQLE

QGKELWREGRVFLQDQNPNRESALKKTHMISMHPITRKDASTSMTMENSLILEDPFECND

SGEDCTRSSTITQCLLTHSGKKPYVSKQCGKSLRNLLSTEPHKQIHTKGKSYQCNLCEKA

YTNCFHLRRHKMTHTGERPYACHLCRKAFTQCSHLRRHEKTHTGQRPYKCHQYGKVFIQS

FNLQRHERTHLGKKCYECDKSGKAFSQSSGFRGNKIIHTGEKPHACLLCGKAFSLSSNLR

>sp|O14727|APAF_HUMAN Apoptotic protease-activating factor 1 OS=Homo sapiens OX=9606 GN=APAF1 PE=1 SV=2

MDAKARNCLLQHREALEKDIKTSYIMDHMISDGFLTISEEEKVRNEPTQQQRAAMLIKMI

LKKDNDSYVSFYNALLHEGYKDLAALLHDGIPVVSSSSGKDSVSGITSYVRTVLCEGGVP

QRPVVFVTRKKLVNAIQQKLSKLKGEPGWVTIHGMAGCGKSVLAAEAVRDHSLLEGCFPG

GVHWVSVGKQDKSGLLMKLQNLCTRLDQDESFSQRLPLNIEEAKDRLRILMLRKHPRSLL

ILDD